# Large Language Model (Few-Shot) for Information Extraction

## Overview

This notebook explores a **few-shot learning approach using Large Language Models (LLMs)** for structured information extraction from clinical trial abstracts in the **EBM-NLP dataset**. The goal is to extract key PIO components—**Population, Intervention, and Outcome**—from unstructured biomedical text.

In contrast to zero-shot prompting, the few-shot approach provides the model with a small number of annotated examples to guide its predictions. These examples help the model better understand the expected output format and the structure of biomedical entities, improving extraction consistency and accuracy.

The pipeline includes:

- Designing prompt templates with a few labeled PIO examples  
- Providing in-context demonstrations to guide the LLM’s reasoning  
- Extracting structured outputs from clinical trial abstracts without model fine-tuning  
- Formatting predictions into a predefined schema for evaluation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import csv
import json
import sys
import ast
import warnings
warnings.filterwarnings("ignore")

In [3]:
!pip install -U bitsandbytes transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 140.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [4]:
!pip install importnb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.8 MB/s eta 0:00:00


In [7]:
import sys
from importnb import Notebook

# Tell Python where the folder is (change this to your actual folder path)
sys.path.append("/content/drive/MyDrive/Colab Notebooks/Few Shot/")

# Now import it like a normal library
with Notebook():
    import Utils

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import userdata

# Load the model and tokenizer
model_id = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit" # This specific version is pre-quantized for speed

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto") # This puts it on the GPU automatically

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

In [9]:
import pandas as pd
df = pd.read_excel(r"/content/drive/MyDrive/Colab Notebooks/cwk_data/cwk_train_data.xlsx")

In [10]:
def extract_few_shot(document_text):
    # This prompt defines exactly what goes into P, I, and O based on your schema
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a strict data extraction tool.
**RULES**:
1. ONLY extract words that are DIRECTLY written in the provided text.
2. If an entity is not explicitly mentioned, return [].
3. DO NOT use external knowledge.
4. DO NOT provide any introductory text or notes.
5. Output MUST be valid JSON only.
6. Participants : Extract Age, Sex, Sample Size, Disease and Condition given in the text.
7. Intervention : Extract name of the drugs which are given in the text, dont manipulate the name of the drugs.
8. Outcome : Extract the outcomes of the medical trial from the text.

### EXAMPLES:

Text: "Naltrexone and communication skills in young children with autism. OBJECTIVE To evaluate the effect of naltrexone on communication skills of young children with autism. METHOD Twenty-four children with autism, 3.0 to 8.3 years old (mean 5.1) who were living at home and attending appropriate school programs, participated in a randomized, double-blind, placebo-controlled, crossover trial. Naltrexone, 1.0 mg/kg, or placebo was administered daily for 2 weeks. Communication was evaluated from videotaped samples of seminaturalistic parent-child interaction. Child and parent language were assessed using similar measures. RESULTS In this heterogeneous sample, the median number of words the child produced on placebo was 9.5 (range 0-124). The median proportion of utterances with echolalia was 0.16. No differences were found between the naltrexone and placebo conditions in any of the measures of children or parents' communication. Significant correlations were found between the child's number of words and developmental quotient (Spearman rho = 0.58, p = .003) and between the child's and parent's number of words (rho = 0.55, p = .005). CONCLUSIONS Previous studies showed that naltrexone was associated with modest reduction in hyperactivity and restlessness in this group of children with autism. In this short-term study, the medication did not lead to improvement in communication, a core deficit of autism."
JSON: {{
  "Participants": ["young children with autism", "Twenty-four", "autism"],
  "Intervention": ["Naltrexone", "placebo", "videotaped samples of seminaturalistic","parent-child", "interaction"],
  "Outcome": ["median number of words the child produced", "median proportion of utterances with echolia", "children or parents' communication", "child's number of words", "developmental quotient"]
  }}

Text: "Beneficial effect of etidronate therapy in chronically hospitalized, disabled patients with stroke. Incidence of hip fractures is high in chronically hospitalized, disabled, elderly patients after stroke. Duration of hospitalization was more than 1 year because of insufficiency of nursing homes. Our study showed that immobilization-induced hypercalcemia and 25-hydroxyvitamin D deficiency contribute to reduced bone mineral density (BMD). This study was designed to address the possibility that treatment with etidronate may reduce the bone resorption and lower the incidence of fractures in elderly patients who are chronically hospitalized and disabled as a result of hemiparesis after stroke. Patients with stroke were randomly assigned to daily treatment with 400 mg of etidronate (n = 40) or a placebo (n = 40), and followed up for 2 years. At baseline, both groups had low BMD with high levels of serum ionized calcium and urinary deoxypyridinoline. In the etidronate group, serum calcium and urinary deoxypyridinoline levels decreased significantly during the study period, whereas the levels in the placebo group were increased. BMD on the hemiplegic side increased by 1.4% in the etidronate group and decreased by 2.2% in the placebo group (P < .001). Two patients sustained hip fractures in the placebo group, and no hip fracture occurred in the etidronate group. Treatment with etidronate increases BMD in chronically hospitalized patients poststroke, and may prevent hip fracture."
JSON: {{
  "Participants": ["chronically hospitalized", "disabled", "stroke", "elderly", "hemiparesis"],
  "Intervention": ["etidronate", "placebo", "etidronate therapy"],
  "Outcome": ["BMD", "high levels of serum ionised calcium", "urinary deoxypyridinoline", "serum calcium", "urinary deoxypyridinoline levels decreased", "BMD on the hemiplegic side increased", "hip fractures", "hip"]
}}

Text: "Xylitol for prevention of acute otitis media."
JSON: {{
  "Participants": ["acute otitis media"],
  "Intervention": ["Xylitol"],
  "Outcome": []
}}

### CURRENT TASK TO DO AND GIVE THE ANSWER IN JSON:
Text: {document_text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # We use temperature 0 for consistent, reproducible results (important for your project)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.0,
        do_sample=False
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    json_string = response.split("assistant")[-1].strip()
    try:
        return json.loads(json_string)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        print(f"Malformed JSON string: {json_string}")
        return {}

In [ ]:
result = extract_few_shot(df["Documents"][5])
print(result)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'Participants': ['patients with asthma', 'asthma'], 'Intervention': ['chlorofluorocarbon (CFC) albuterol', 'hydrofluoroalkane-134a (HFA) albuterol'], 'Outcome': ['bronchodilator efficacy', 'pulse rate', 'blood pressure', 'ECG intervals', 'adverse events', 'asthma symptoms', 'rhinitis', 'laboratory tests', 'physical examinations']}


In [13]:
file_path = r"/content/drive/MyDrive/Colab Notebooks/Few Shot/result_few_shot_2.csv"

In [14]:
Utils.config(df, file_path, 200, extract_few_shot )

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Finished and saved Doc ID: 10036953


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Finished and saved Doc ID: 10037531


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Finished and saved Doc ID: 11013281


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Finished and saved Doc ID: 11013775


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Finished and saved Doc ID: 11015817
Finished and saved Doc ID: 11021553
